In [1]:
import spacy
from spacy.language import Language

In [2]:
nlp = spacy.load("en_core_web_sm") # carregando o modelo

In [12]:
with open("historico_wow.txt", "r") as f:
    text = f.read()

doc = nlp(text)

# verificando o que o modelo reconhece antes de qualquer modificação
for ent in doc.ents:
    print(ent.text, ent.label_)

the World First Kill of Heroic ORG
Lich King PERSON
Paragon ORG
Paladin - 1 Protection ORG
2 Holy PERSON
Lich King Strategy PERSON
Paragon PERSON
Lich King PERSON
Frost DK PERSON
Boomkin GPE
Lich King PERSON
Val'kyr Shadowguard ORG
3M Health Points
Raging Spirits ORG
Ghouls PERSON
Protection Paladin PERSON
Lazeil GPE
Ghouls PERSON
Lazeil’s PERSON
Lazeil PERSON
Ghouls PERSON
Sejta GPE
Protection Paladin PERSON
Ardent Defender PERSON
Lazeil PERSON
Enrage ORG
Retribution Paladin PERSON
Discipline Priest PERSON
Enrage ORG
Spirits PRODUCT
Druid ORG
Sejta PRODUCT
Frozen Orbs PERSON
Rogues ORG
Spirit ORG
Boomkin GPE
Spirit ORG
Lazeil PERSON
Holy Paladins PERSON
Ilonie PERSON
Diamondtear ORG
Hammer of Justice ORG
Paladins PERSON
Retribution Paladins PERSON
Paladins PERSON
Holy Wrath PERSON
Rogues ORG
Kidney Shot PERSON
an Unholy DK’s Desecration ORG
Crippling Poison PERSON
Retribution Paladin PERSON
Righteous Fury GPE
Infest PERSON
DEFILE GPE
Defile PERSON
Defile GPE
Defile GPE
Melee PERSON
De

In [13]:
WOW_ENTITIES = {"PERSON", "ORG", "GPE", "PRODUCT"} # tipos que fazem sentido no contexto

@Language.component("wow_filter")
def wow_filter(doc):
    doc.ents = [ent for ent in doc.ents if ent.label_ in WOW_ENTITIES] # mantem so o que interessa
    return doc

In [22]:
if "wow_filter" not in nlp.pipe_names:
    nlp.add_pipe("wow_filter") # adicionando o componente
nlp.analyze_pipes() # confirmando a posição do componente

{'summary': {'tok2vec': {'assigns': ['doc.tensor'],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'tagger': {'assigns': ['token.tag'],
   'requires': [],
   'scores': ['tag_acc',
    'pos_acc',
    'tag_micro_p',
    'tag_micro_r',
    'tag_micro_f'],
   'retokenizes': False},
  'parser': {'assigns': ['token.dep',
    'token.head',
    'token.is_sent_start',
    'doc.sents'],
   'requires': [],
   'scores': ['dep_uas',
    'dep_las',
    'dep_las_per_type',
    'sents_p',
    'sents_r',
    'sents_f'],
   'retokenizes': False},
  'attribute_ruler': {'assigns': [],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'lemmatizer': {'assigns': ['token.lemma'],
   'requires': [],
   'scores': ['lemma_acc'],
   'retokenizes': False},
  'ner': {'assigns': ['doc.ents', 'token.ent_iob', 'token.ent_type'],
   'requires': [],
   'scores': ['ents_f', 'ents_p', 'ents_r', 'ents_per_type'],
   'retokenizes': False},
  'wow_filter': {'assigns': [],
   'requires': [],
   

In [16]:
doc2 = nlp(text) # reprocessando com o filtro ativo

for ent in doc2.ents: # apenas entidades dos tipos relevantes
    print(ent.text, ent.label_)

the World First Kill of Heroic ORG
Lich King PERSON
Paragon ORG
Paladin - 1 Protection ORG
2 Holy PERSON
Lich King Strategy PERSON
Paragon PERSON
Lich King PERSON
Frost DK PERSON
Boomkin GPE
Lich King PERSON
Val'kyr Shadowguard ORG
3M Health Points
Raging Spirits ORG
Ghouls PERSON
Protection Paladin PERSON
Lazeil GPE
Ghouls PERSON
Lazeil’s PERSON
Lazeil PERSON
Ghouls PERSON
Sejta GPE
Protection Paladin PERSON
Ardent Defender PERSON
Lazeil PERSON
Enrage ORG
Retribution Paladin PERSON
Discipline Priest PERSON
Enrage ORG
Spirits PRODUCT
Druid ORG
Sejta PRODUCT
Frozen Orbs PERSON
Rogues ORG
Spirit ORG
Boomkin GPE
Spirit ORG
Lazeil PERSON
Holy Paladins PERSON
Ilonie PERSON
Diamondtear ORG
Hammer of Justice ORG
Paladins PERSON
Retribution Paladins PERSON
Paladins PERSON
Holy Wrath PERSON
Rogues ORG
Kidney Shot PERSON
an Unholy DK’s Desecration ORG
Crippling Poison PERSON
Retribution Paladin PERSON
Righteous Fury GPE
Infest PERSON
DEFILE GPE
Defile PERSON
Defile GPE
Defile GPE
Melee PERSON
De

In [17]:
known_players = {"Lazeil", "Sejta", "Lappé", "Synti", "Jhazrun", "Ilonie", "Diamondtear"} # jogadores do Paragon citados no texto

@Language.component("tag_players")
def tag_players(doc):
    original_ents = list(doc.ents)
    new_ents = []
    for ent in original_ents:
        if ent.text in known_players: # verifica se a entidade é um jogador conhecido
            new_ent = doc.char_span(ent.start_char, ent.end_char, label="PLAYER") # cria uma nova entidade com rótulo customizado
            if new_ent:
                new_ents.append(new_ent)
        else:
            new_ents.append(ent)
    doc.ents = new_ents
    return doc

In [18]:
# adicionando o segundo componente depois do filtro
nlp.add_pipe("tag_players", after="wow_filter")
nlp.analyze_pipes() # confirmando a ordem dos dois componentes no pipeline

{'summary': {'tok2vec': {'assigns': ['doc.tensor'],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'tagger': {'assigns': ['token.tag'],
   'requires': [],
   'scores': ['tag_acc',
    'pos_acc',
    'tag_micro_p',
    'tag_micro_r',
    'tag_micro_f'],
   'retokenizes': False},
  'parser': {'assigns': ['token.dep',
    'token.head',
    'token.is_sent_start',
    'doc.sents'],
   'requires': [],
   'scores': ['dep_uas',
    'dep_las',
    'dep_las_per_type',
    'sents_p',
    'sents_r',
    'sents_f'],
   'retokenizes': False},
  'attribute_ruler': {'assigns': [],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'lemmatizer': {'assigns': ['token.lemma'],
   'requires': [],
   'scores': ['lemma_acc'],
   'retokenizes': False},
  'ner': {'assigns': ['doc.ents', 'token.ent_iob', 'token.ent_type'],
   'requires': [],
   'scores': ['ents_f', 'ents_p', 'ents_r', 'ents_per_type'],
   'retokenizes': False},
  'wow_filter': {'assigns': [],
   'requires': [],
   

In [20]:
doc3 = nlp(text) # reprocessando com os dois componentes ativos

for ent in doc3.ents:
    print(ent.text, ent.label_) # jogadores aparecem agora com rotulo

the World First Kill of Heroic ORG
Lich King PERSON
Paragon ORG
Paladin - 1 Protection ORG
2 Holy PERSON
Lich King Strategy PERSON
Paragon PERSON
Lich King PERSON
Frost DK PERSON
Boomkin GPE
Lich King PERSON
Val'kyr Shadowguard ORG
3M Health Points
Raging Spirits ORG
Ghouls PERSON
Protection Paladin PERSON
Lazeil PLAYER
Ghouls PERSON
Lazeil’s PERSON
Lazeil PLAYER
Ghouls PERSON
Sejta PLAYER
Protection Paladin PERSON
Ardent Defender PERSON
Lazeil PLAYER
Enrage ORG
Retribution Paladin PERSON
Discipline Priest PERSON
Enrage ORG
Spirits PRODUCT
Druid ORG
Sejta PLAYER
Frozen Orbs PERSON
Rogues ORG
Spirit ORG
Boomkin GPE
Spirit ORG
Lazeil PLAYER
Holy Paladins PERSON
Ilonie PLAYER
Diamondtear PLAYER
Hammer of Justice ORG
Paladins PERSON
Retribution Paladins PERSON
Paladins PERSON
Holy Wrath PERSON
Rogues ORG
Kidney Shot PERSON
an Unholy DK’s Desecration ORG
Crippling Poison PERSON
Retribution Paladin PERSON
Righteous Fury GPE
Infest PERSON
DEFILE GPE
Defile PERSON
Defile GPE
Defile GPE
Melee P

In [21]:
import os

os.makedirs("data", exist_ok=True)
nlp.to_disk("data/wow_nlp") # salvando o modelo customizado com os dois componentes